# ZemaVoice Stage 6 — Long-Form Amharic Narration
Paste long Amharic scripts. Separate scenes with blank lines. Generate individual scene WAVs, one combined WAV, and a timing JSON inside a ZIP.

**Important:** 5/10/15-minute options are *reference targets*, not guaranteed durations. Audio is generated at natural length without artificial silence to fill minutes. Long scripts may exhaust Colab runtime resources; save exports after each run.

**License:** This notebook uses Meta MMS Amharic under CC BY-NC 4.0 for noncommercial evaluation only. Do not deploy it in a commercial ZemaVoice product.

In [ ]:
!pip -q install 'transformers>=4.44,<5' 'accelerate>=0.26' 'soundfile>=0.12' 'uroman>=1.3' 'ipywidgets>=8'


In [ ]:
import re, json, zipfile, torch, numpy as np, soundfile as sf
from pathlib import Path
from IPython.display import Audio, display
from transformers import VitsModel, VitsTokenizer, set_seed
from uroman import Uroman
import ipywidgets as widgets
model_id='facebook/mms-tts-amh'
tokenizer=VitsTokenizer.from_pretrained(model_id)
model=VitsModel.from_pretrained(model_id).eval()
romanizer=Uroman()
device='cuda' if torch.cuda.is_available() else 'cpu'
model=model.to(device)
sr=model.config.sampling_rate
print('Ready:',device,'| Sample rate:',sr)


In [ ]:
script=widgets.Textarea(value='በአንድ ጨለማ ሌሊት፣ መንደሩ በጸጥታ ተውጦ ነበር።\n\nበሩቅ ከሚታየው ተራራ በስተጀርባ፣ የሚያስደንቅ ብርሃን ታየ።',description='Script:',layout=widgets.Layout(width='95%',height='250px'))
target=widgets.Dropdown(options=[('Natural length',0),('5 minutes (reference)',300),('10 minutes (reference)',600),('15 minutes (reference)',900)],description='Length:')
pause_ms=widgets.IntSlider(value=480,min=0,max=2000,step=40,description='Pause ms:')
section_gap_ms=widgets.IntSlider(value=800,min=0,max=3000,step=100,description='Section gap:')
seed=widgets.IntText(value=42,description='Seed:')
display(script,target,pause_ms,section_gap_ms,seed)
print('Separate scenes with a blank line. Target length is a reference only; the generator does not pad or stretch audio.')


In [ ]:
raw=script.value.strip()
assert raw, 'Paste an Amharic script first.'
assert len(raw)<=120000, 'Use at most 120,000 characters per run; split longer scripts.'
sections=[s.strip() for s in re.split(r'\n\s*\n+',raw) if s.strip()]
assert len(sections)<=100, 'Use at most 100 scenes per run.'
assert len(sections)>0
out=Path('/content/zemavoice_stage6');out.mkdir(exist_ok=True)
set_seed(int(seed.value))
manifest=[];full=[];position=0
for section_index,section in enumerate(sections,1):
    sentences=[s.strip() for s in re.findall(r'[^።!?\n]+[።!?]?',section) if s.strip()]
    if not sentences: continue
    scene=[]
    for sentence_index,sentence in enumerate(sentences,1):
        romanized=romanizer.romanize_string(sentence)
        if not romanized: raise ValueError(f'Romanization failed: scene {section_index}, sentence {sentence_index}')
        inputs=tokenizer(romanized,return_tensors='pt').to(device)
        with torch.inference_mode():
            audio=model(**inputs).waveform[0].detach().cpu().float().numpy()
        scene.append(audio)
        if sentence_index<len(sentences) and pause_ms.value:
            scene.append(np.zeros(round(sr*pause_ms.value/1000),dtype=np.float32))
    audio=np.concatenate(scene)
    name=f'scene_{section_index:03d}.wav'
    sf.write(out/name,audio,sr)
    manifest.append({'scene':section_index,'filename':name,'start_seconds':round(position/sr,3),'duration_seconds':round(len(audio)/sr,3),'text':section})
    full.append(audio);position+=len(audio)
    if section_index<len(sections) and section_gap_ms.value:
        gap=np.zeros(round(sr*section_gap_ms.value/1000),dtype=np.float32)
        full.append(gap);position+=len(gap)
    print(f'Scene {section_index}: {len(audio)/sr:.1f} seconds')
combined=np.concatenate(full)
sf.write(out/'full_narration.wav',combined,sr)
(out/'scene_timing.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')
actual=len(combined)/sr
print(f'Total duration: {actual:.2f} seconds ({actual/60:.2f} minutes)')
if target.value:
    print(f'Reference target: {target.value/60:.0f} minutes; difference: {(actual-target.value):+.1f} seconds. Add or shorten script to change natural spoken duration.')
display(Audio(combined,rate=sr))


In [ ]:
from google.colab import files
archive=Path('/content/zemavoice_stage6_export.zip')
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for f in sorted(out.glob('*')):z.write(f,arcname=f.name)
print('Export includes full_narration.wav, individual scene WAVs, and scene_timing.json')
files.download(str(archive))
